# 02 – Baselines + evaluation (Phase 3)

Before any real model, we score dead-simple predictions on 2024 val. Every later model has to beat them.
Uses the locked target from Phase 2 (`src/features/target.py`). Run `.venv/bin/python -m src.data.load` once first if `data/raw/` is empty.

In [ ]:
import sys
from pathlib import Path

# Let the notebook import code from src/ (the notebook lives one folder down).
sys.path.insert(0, str(Path.cwd().parent))

import polars as pl
from src.data.load import load_pbp
from src.features.qb_games import qb_games
from src.features.target import build_targets

pl.Config.set_tbl_cols(30)
from src.models.baselines import league_average, predict_constant
from src.models.evaluate import mae

In [ ]:
# Train + val + test seasons only (2016-2025). We touch val to score; test stays untouched.
pbp = load_pbp(list(range(2016, 2026)))
games = qb_games(pbp)           # history: every QB-game, any size
targets = build_targets(games)  # what we predict and score
train = targets.filter(pl.col("split") == "train")
val = targets.filter(pl.col("split") == "val")
train.height, val.height

## Concept 1: What a baseline is

A baseline is the dumbest prediction that's still honest. It's the ruler a real model gets measured against:
if a fancy model can't beat "guess the league average for everyone", it hasn't learned anything.

The dumbest one: every QB, every week, gets the same rating — the **train** league average
(dropback-weighted). It's computed from 2016–2023 only, so it uses nothing from 2024.

In [ ]:
# The one number: train league-average EPA/dropback, weighted by dropbacks.
league_avg = (train["target"] * train["weight"]).sum() / train["weight"].sum()
print(f"league average (train, weighted): {league_avg:+.3f}")

# Worked example: how wrong that one number is, game by game, for two very different 2023 QBs.
example = (
    train.filter(pl.col("season") == 2023, pl.col("passer").is_in(["P.Mahomes", "Z.Wilson"]), pl.col("week") <= 3)
    .with_columns(pl.lit(league_avg).alias("prediction"))
    .with_columns((pl.col("target") - pl.col("prediction")).abs().alias("error"))
    .select("passer", "week", "target", "prediction", "error", "weight")
    .sort("passer", "week")
)
example

### Score it on 2024 val
The number every later model has to beat. Logged in experiments.md as the first row.

In [ ]:
# Step 1: one number from train. Step 2: give it to every val game. Step 3: score.
league_avg = league_average(train)
baseline_scores = mae(predict_constant(val, league_avg))
print(f"league-average baseline (+{league_avg:.3f} for everyone), 2024 val:")
print(f"  weighted MAE   {baseline_scores['weighted']:.3f}   <- official score")
print(f"  unweighted MAE {baseline_scores['unweighted']:.3f}")

### ▶ Play: Concept 1
Change the one number everyone gets, and see how the MAE moves. Then pick a QB and see how wrong "everyone is average" is for him.

In [ ]:
# ---- KNOBS -----------------------------------------------------------------
PREDICTION = 0.0        # Try 0.0, 0.2, -0.1, or 0.074 (the 2024 median). How much does MAE move?
SHOW_QB = "L.Jackson"   # Try "J.Goff", "B.Young", "W.Levis". Who does the baseline get most wrong?
# ---------------------------------------------------------------------------

now = mae(predict_constant(val, PREDICTION))
for name, s in [(f"default (+{league_avg:.3f})", baseline_scores), (f"now ({PREDICTION:+.3f})", now)]:
    print(f"{name:20s} weighted MAE {s['weighted']:.4f}   unweighted MAE {s['unweighted']:.4f}")
print(f"change: weighted {now['weighted'] - baseline_scores['weighted']:+.4f}")

# One QB's 2024: his actual games vs the number everyone gets.
qb = predict_constant(val.filter(pl.col("passer") == SHOW_QB), PREDICTION)
if qb.height == 0:
    print(f"\nNo 2024 games for {SHOW_QB}. Names look like 'P.Mahomes'.")
else:
    s = mae(qb)
    print(f"\n{SHOW_QB} 2024: {qb.height} games, actual avg {qb['target'].mean():+.3f}, "
          f"weighted MAE with {PREDICTION:+.3f} for everyone: {s['weighted']:.3f}")

## Your exploration

Add cells below.